# Hip Bone Reconstruction — End-to-end Colab Notebook

Single notebook that covers the whole pipeline:

1. Set up the environment (clone repo, install deps)
2. Mount Google Drive (so your data + checkpoint persist across sessions)
3. Get a hip-CT dataset and prepare it (CTPelvic1K → 128³ `.npy`)
4. Train a 3D U-Net on synthetic defects and save the checkpoint
5. Upload your own patient scan (`.nii.gz` from TotalSegmentator)
6. Reconstruct the missing piece, view it inline, and download the STL

**Before you start:** switch the runtime to GPU — *Runtime → Change runtime type → GPU* (T4 is fine).

## 1. GPU sanity check

In [1]:
!nvidia-smi || echo 'No GPU detected — switch to a GPU runtime: Runtime > Change runtime type > GPU'

Sat Apr 25 23:36:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo and install dependencies

In [2]:
%cd /content
![ -d Claude-hip-try1 ] || git clone -b claude/hip-bone-reconstruction-bGrN1 https://github.com/bhaskarsdose/Claude-hip-try1.git
%cd /content/Claude-hip-try1
!git pull
!pip install -q -r requirements.txt
!pip install -q -e .

/content
Cloning into 'Claude-hip-try1'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 7), reused 45 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 37.45 KiB | 639.00 KiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Claude-hip-try1
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 56.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hip_recon (pyproject.toml) ... done


## 3. Mount Google Drive (recommended)

Drive is used for two things: (1) holding the raw CTPelvic1K data so you don't re-download it on every Colab session, and (2) saving the trained checkpoint. If you skip this, everything stays in `/content/` and is lost when the runtime recycles.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/hip-recon'
os.makedirs(DRIVE, exist_ok=True)
print('Drive workspace:', DRIVE)

Mounted at /content/drive
Drive workspace: /content/drive/MyDrive/hip-recon


## 4. Get a hip-CT dataset

We use **[CTPelvic1K](https://zenodo.org/records/4588403)** — the public benchmark for pelvis segmentation. The next cells download the **seven mask archives** (~227 MB total) directly from Zenodo. We do *not* download the raw CT volumes (`dataset6_data.tar.gz`, `dataset7_data.tar.gz` — together ~15 GB), because for hip-shape completion the network only needs the binary bone masks, not Hounsfield intensities.

If you already have hip masks elsewhere on Drive, set `DATA_RAW` below to that folder and skip the download cell.

In [ ]:
import os
DATA_RAW = '/content/drive/MyDrive/hip-recon/raw'              # NIfTI hip masks live here
DATA_PROCESSED = '/content/drive/MyDrive/hip-recon/processed'  # 128**3 .npy training tensors
os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)
# Export to the shell environment so subsequent !-cells (preprocess, train) see them.
os.environ['DATA_RAW'] = DATA_RAW
os.environ['DATA_PROCESSED'] = DATA_PROCESSED
print('DATA_RAW      =', DATA_RAW)
print('DATA_PROCESSED=', DATA_PROCESSED)

In [5]:
%%writefile urls.txt
# CTPelvic1K mask archives (Zenodo record 4588403). ~227 MB total.
# Skip the corresponding download cell if you already have masks on Drive.
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset1_mask_mappingback.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset2_mask_mappingback.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset3_mask_mappingback.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset4_mask_mappingback.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset5_mask_mappingback.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset6_Anonymized_mask.tar.gz
https://zenodo.org/records/4588403/files/CTPelvic1K_dataset7_mask.tar.gz

Writing urls.txt


In [ ]:
# Download (skips files already on Drive) and extract every CTPelvic1K
# mask archive. Pure Python so it doesn't depend on shell-variable
# expansion — that bit us last time.
import subprocess, glob, os, sys

print('Downloading mask archives...')
subprocess.run(
    [sys.executable, 'scripts/download_ctpelvic1k.py',
     '--urls-from', 'urls.txt', '--out', DATA_RAW],
    check=True,
)

tarballs = sorted(glob.glob(os.path.join(DATA_RAW, '*.tar.gz')))
print(f'\nExtracting {len(tarballs)} tarballs into {DATA_RAW}')
for t in tarballs:
    print(f'  {os.path.basename(t)}')
    subprocess.run(['tar', '-xzf', t, '-C', DATA_RAW], check=True)

masks = glob.glob(os.path.join(DATA_RAW, '**', '*.nii.gz'), recursive=True)
print(f'\nFound {len(masks)} .nii.gz mask files. First few:')
for m in masks[:5]:
    print(f'  {m}')

## 5. Preprocess: NIfTI → 128³ `.npy`

For each mask in `DATA_RAW`: binarise the requested label, resample to isotropic 1.5 mm voxels, crop, PCA-canonicalise the pose, and save as a 128³ `.npy`.

CTPelvic1K labels: **1=sacrum, 2=L-hip, 3=R-hip, 4=L5**. We train on **left hip only** (`--label 2`).

> **If you previously ran this cell without `--label 2`** (whole-pelvis data), clear the processed directory first so stale files don't mix with the left-hip tensors:
> ```python
> import shutil; shutil.rmtree(DATA_PROCESSED, ignore_errors=True)
> import os; os.makedirs(DATA_PROCESSED, exist_ok=True)
> ```

In [ ]:
# If switching from whole-pelvis to left-hip (label 2), clear stale tensors first:
# import shutil; shutil.rmtree(DATA_PROCESSED, ignore_errors=True)
# import os; os.makedirs(DATA_PROCESSED, exist_ok=True)

# --workers 4 runs 4 files in parallel (3-4x faster). Already-done files are skipped
# automatically so you can safely re-run this cell if it was interrupted.
!python scripts/prepare_dataset.py \
    --in-dir $DATA_RAW \
    --out-dir $DATA_PROCESSED \
    --label 2 \
    --workers 4
!echo "--- processed files ---"
!ls $DATA_PROCESSED | wc -l

## 6. Train

Trains a MONAI 3D U-Net on `(defective, complete)` pairs synthesised from the prepared volumes. Best checkpoint goes to `models/unet3d_hip.pt`; we also copy it to Drive so the next inference run can pick it up without retraining.

**Speed tuning by GPU runtime**

| Runtime | `--batch-size` | Speedup vs. T4 bs=2 |
|---|---|---|
| T4 (free / Pro) | `4` | 2× (with AMP) |
| L4 (Pro) | `8` | ~4× |
| V100 16GB (Pro) | `6` | ~3× |
| A100 40GB (Pro+) | `16` | ~8–10× |

If you hit a CUDA OOM error, halve `--batch-size`. AMP is on by default; it's ~1.7× faster than `--no-amp`. Increase `--num-workers` to 4 if Drive I/O is the bottleneck.

In [ ]:
# --no-amp: disables mixed precision. AMP is faster but on a 3D U-Net at
# lr=1e-3 it triggers NaN cascades (seen at step 174 in this dataset).
# Without AMP the model runs in fp32 — slightly more memory, no instability.
# Convergence: train_loss ~0.1 by epoch 5, val_dice 0.9+ by epoch 10.
# Side-cell GPU check: !nvidia-smi --query-gpu=utilization.gpu --format=csv -l 2
!mkdir -p models
!python -m hip_recon.train \
    --data-dir $DATA_PROCESSED \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-3 \
    --num-workers 4 \
    --no-amp \
    --out models/unet3d_hip.pt

### 6b. Fine-tune for massive defects (optional)

If your patient case has the bone split into **two disconnected pieces with a big middle gap** (pelvic discontinuity / massive osteolysis), the base model may treat each piece as already complete and predict an empty implant.

The fix is to fine-tune with an additional "slab" defect type — chunks removed from the middle of the bone, leaving two end pieces. The next cell takes ~15 min on A100 and starts from the existing dice-0.94 checkpoint.

In [ ]:
# Optional: fine-tune the existing checkpoint to handle severe defects
# (pelvic discontinuity / massive osteolysis where the bone is in two
# disconnected pieces with a big middle gap).
#
# This loads your dice-0.94 checkpoint and trains 20 more epochs with the
# new "slab" defect type added to the mix. Lower lr (3e-4) so we specialise
# without forgetting the small-defect skill. Takes ~15 min on A100.
#
# Skip this cell if your STLs only have small/medium defects — the original
# 50-epoch training already handles those.
!python -m hip_recon.train \
    --data-dir $DATA_PROCESSED \
    --epochs 20 \
    --batch-size 8 \
    --lr 3e-4 \
    --num-workers 4 \
    --no-amp \
    --resume models/unet3d_hip.pt \
    --out models/unet3d_hip.pt

In [ ]:
import shutil
shutil.copy2('models/unet3d_hip.pt', f'{DRIVE}/unet3d_hip.pt')
print('Saved checkpoint to', f'{DRIVE}/unet3d_hip.pt')

In [ ]:
# Optional: live training curves
%load_ext tensorboard
%tensorboard --logdir runs

---
# Inference — reconstruct a missing piece from your own scan

From this point on you can re-run the notebook **without retraining**: as long as `models/unet3d_hip.pt` exists locally (or in Drive), inference works.

If you've restarted the runtime since training, run the next cell to copy the checkpoint back from Drive.

In [ ]:
import os, shutil
os.makedirs('models', exist_ok=True)
src = f'{DRIVE}/unet3d_hip.pt'
if os.path.isfile(src) and not os.path.isfile('models/unet3d_hip.pt'):
    shutil.copy2(src, 'models/unet3d_hip.pt')
    print('Restored checkpoint from Drive.')
elif os.path.isfile('models/unet3d_hip.pt'):
    print('Local checkpoint already present.')
else:
    print('WARNING: no checkpoint found. Inference will run in placeholder mode.')

## 7. Upload your patient scan

Upload your **defective** left-hip bone in either format:

- **NIfTI mask** (`.nii.gz` / `.nii`) — the `hip_left.nii.gz` produced by [TotalSegmentator](https://github.com/wasserth/TotalSegmentator) from the patient's CT.
- **STL mesh** (`.stl`) — a triangle mesh of the defective hip bone (e.g. exported from 3D Slicer or Mimics).

The reconstruction code auto-detects the format from the file extension.

In [ ]:
from google.colab import files
uploaded = files.upload()
patient_path = next(iter(uploaded))
print('Uploaded:', patient_path)

## 8. Run reconstruction

In [ ]:
from hip_recon.infer import reconstruct
result = reconstruct(patient_path)   # auto-detects .nii.gz or .stl
print('Trained model used :', result.used_trained_model)
print('Input mesh         :', len(result.input_mesh.vertices), 'verts',
      '/', len(result.input_mesh.faces), 'faces')
print('Implant mesh       :', len(result.implant_mesh.vertices), 'verts',
      '/', len(result.implant_mesh.faces), 'faces')

## 9. View the reconstruction inline

Plotly renders the input bone (gray) and the predicted implant (blue) in the same scene. Drag to rotate, scroll to zoom.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def mesh_trace(mesh, color, name, opacity=0.85):
    if len(mesh.vertices) == 0:
        return None
    v = np.asarray(mesh.vertices)
    f = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color, opacity=opacity, name=name,
        showlegend=True, flatshading=False,
    )

traces = [t for t in [
    mesh_trace(result.input_mesh,   '#b8c0cc', 'Input bone',     opacity=0.55),
    mesh_trace(result.implant_mesh, '#6aa6ff', 'Predicted implant', opacity=0.95),
] if t is not None]

fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(aspectmode='data', xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    height=720, margin=dict(l=0, r=0, t=30, b=0),
    title='Hip reconstruction — input bone + predicted implant',
)
fig.show()

## 10. Save STLs and download

`implant.stl` is the file to send to your slicer / 3D printer.

In [ ]:
os.makedirs('outputs', exist_ok=True)
input_stl = 'outputs/input.stl'
implant_stl = 'outputs/implant.stl'
result.input_mesh.export(input_stl)
result.implant_mesh.export(implant_stl)
print('Wrote', input_stl, implant_stl)
files.download(input_stl)
files.download(implant_stl)

---

**Tips**

- To reconstruct a different patient, re-run cells 7–10 only — no need to retrain.
- The implant mesh is in the model's canonical pose, not patient space. For surgical placement you'll need to register it back to the original CT — that's a follow-up step.
- If the implant looks wrong: increase `--epochs` in cell 6, prep more cases (cell 5), or train on a single hip side via `--label 2` / `--label 3`.
- Empty implant means the model thinks the input is already complete — verify the input mask actually has a defect.

---
# Mirror reconstruction — recommended for unilateral defects

If the patient has **one healthy hip and one defective hip**, mirroring the healthy contralateral side is the gold-standard reconstruction approach (uses the patient's own anatomy as ground truth — beats any AI model on severe defects).

**One cell does everything**: upload defective + healthy STL, reconstruct, view, download.

In [ ]:
# =====================================================================
# ALL-IN-ONE: upload both hips → reconstruct → preview → download.
# Just run this single cell. Self-heals "No module named hip_recon".
# =====================================================================
import sys, os, subprocess

# Self-heal: if the package isn't visible to the kernel (common after a
# Colab runtime restart), clone the repo if missing and add src/ to the path.
REPO = '/content/Claude-hip-try1'
if not os.path.isdir(REPO):
    subprocess.run(
        ['git', 'clone', '-b', 'claude/hip-bone-reconstruction-bGrN1',
         'https://github.com/bhaskarsdose/Claude-hip-try1.git', REPO],
        check=True,
    )
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import importlib, hip_recon.mirror
importlib.reload(hip_recon.mirror)
from hip_recon.mirror import mirror_reconstruct
from google.colab import files
import numpy as np
import plotly.graph_objects as go

# 1) Upload the two STLs (defective hip first, healthy contralateral second)
print('1) Choose the DEFECTIVE hip STL...')
defective_path = next(iter(files.upload()))
print('   defective:', defective_path)
print('\n2) Choose the HEALTHY contralateral hip STL...')
healthy_path = next(iter(files.upload()))
print('   healthy:  ', healthy_path)

# 2) Reconstruct
mres = mirror_reconstruct(
    defective_path, healthy_path,
    mirror_axis=0,        # try 1 or 2 if implant ends up on wrong side
    dilate_input=0,       # 0 = snug fit, 1 = small cement gap, 3 = safe margin
    cleanup_iters=3,      # 3 severs thin "fingers"; 4-5 = more aggressive
    smooth_iters=15,
    keep_largest=True,
)
print(f'\nImplant: {len(mres.implant_mesh.vertices)} verts / '
      f'{len(mres.implant_mesh.faces)} faces')

# 3) Inline 3D preview
def trace(m, c, n, op=0.85):
    if len(m.vertices) == 0: return None
    v, f = np.asarray(m.vertices), np.asarray(m.faces)
    return go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2],
                     i=f[:,0], j=f[:,1], k=f[:,2],
                     color=c, opacity=op, name=n, showlegend=True)

fig = go.Figure(data=[t for t in [
    trace(mres.defective_mesh, '#b8c0cc', 'Defective bone', 0.55),
    trace(mres.implant_mesh,   '#ff9b3d', 'Implant',         0.95),
] if t])
fig.update_layout(scene=dict(aspectmode='data'), height=720,
                  title='Hip implant — mirror reconstruction')
fig.show()

# 4) Save + download
os.makedirs('outputs', exist_ok=True)
out = 'outputs/implant.stl'
mres.implant_mesh.export(out)
print(f'\nSaved {out}')
files.download(out)